# `PEPit.jl` tutorial: Accelerated gradient method for smooth strongly convex minimization

## Setup
Before we do anything let us load the necessary packages.

In [13]:
using PEPit, OrderedCollections, Mosek, MosekTools, Clarabel

Let us start with defining an empty PEP, which we will populate step by step.

In [14]:
problem = PEP()

PEP(AbstractFunction[], Point[], Constraint[], Expression[], PSDMatrix[], nothing)

## Problem in consideration

Consider the convex minimization problem
$$f_\star \triangleq \min_x f(x),$$
where $f$ is $L$-smooth and $\mu$-strongly convex. For this example, let us take $\mu=0.1$ and $L=1$. Let us declare this function type in `PEPit`. 

In [15]:
μ = 0.1
L = 1
param = OrderedDict("mu" => μ, "L" => L)

OrderedDict{String, Real} with 2 entries:
  "mu" => 0.1
  "L"  => 1

This defines the set of parameters $\mu$ and $L$ to describe the function class. Okay, now time to define the function class itself.  

In [16]:
func = declare_function!(problem, SmoothStronglyConvexFunction, param; reuse_gradient=true) # creates the class of strongly convex and smooth functions with parameters $\mu = 0.1$ and $L=1$

SmoothStronglyConvexFunction(0.1, 1.0, PEPFunction(1, true, OrderedDict{PEPFunction, Float64}(PEPFunction(#= circular reference @-2 =#) => 1.0), true, Tuple{Point, Point, Expression}[], Tuple{Point, Point, Expression}[], Constraint[], PSDMatrix[], PSDMatrix[], 0))

## Algorithm

What we want to study in this tutorial is to compute the a worst-case guarantee for the  **fast gradient** method described as follows. 

### Algorithm:
For $t \in \{0, \dots, n-1\}$,


$$\begin{aligned}
    y_t & = x_t + \frac{\sqrt{L} - \sqrt{\mu}}{\sqrt{L} + \sqrt{\mu}}(x_t - x_{t-1}) \\
    x_{t+1} & = y_t - \frac{1}{L} \nabla f(y_t)
\end{aligned}$$
with $x_{-1}:= x_0$.

## Goal

We want to compute the smallest possible $\tau(n, L, \mu)$ such that the guarantee
$$f(x_n) - f_\star \leqslant \tau(n, L, \mu) \left(f(x_0) -  f(x_\star) + \frac{\mu}{2}\|x_0 - x_\star\|^2\right),$$
is valid, where $x_n$ is the output of the **fast gradient** method, and where $x_\star$ is the minimizer of $f$. Sometimes we say that $\left(f(x_0) -  f(x_\star) + \frac{\mu}{2}\|x_0 - x_\star\|^2\right)$ is the initial condition for this PEP. In short, for given values of $n$, $L$ and $\mu$,
$\tau(n, L, \mu)$ is computed as the worst-case value of
$f(x_n)-f_\star$ when $f(x_0) -  f(x_\star) + \frac{\mu}{2}\|x_0 - x_\star\|^2 \leqslant 1$.

## Populate the PEP step by step

Let us implement the algorithm in PEPit now. For this intorductory example we will let $n=2$.

In [17]:
n = 2

2

### Define $x_0, x_\star, f_\star$
We start with defining the initial point $x_0 \equiv $ `x0`, opitmal point $x_\star \equiv$ `xs`, and the optimal objective value $f_\star \equiv$ `fs`.

In [18]:
xs = stationary_point!(func) # creates $x_\star$

fs = value!(func, xs) # computes $f_\star = f(x_\star)$

x0 = set_initial_point!(problem) # creates $x_0$

Point(5, true, OrderedDict{Point, Float64}(Point(#= circular reference @-2 =#) => 1.0), 1, nothing)

### Define initial condition

The next step is now define our initial condition, which is $f(x_0) -  f(x_\star) + \frac{\mu}{2}\|x_0 - x_\star\|^2 \leq 1$. 

In [19]:
set_initial_condition!(problem, value!(func, x0) - fs + μ / 2 * (x0 - xs)^2 <= 1) # creates the initial condition $f(x_0) -  f(x_\star) + \frac{\mu}{2}\|x_0 - x_\star\|^2 \leq 1$

1-element Vector{Constraint}:
 Constraint(Expression(15, false, OrderedDict{Any, Float64}(Expression(6, true, OrderedDict{Any, Float64}(Expression(#= circular reference @-2 =#) => 1.0), 1, nothing) => 1.0, Expression(4, true, OrderedDict{Any, Float64}(Expression(#= circular reference @-2 =#) => 1.0), 0, nothing) => -1.0, (Point(5, true, OrderedDict{Point, Float64}(Point(#= circular reference @-2 =#) => 1.0), 1, nothing), Point(5, true, OrderedDict{Point, Float64}(Point(#= circular reference @-2 =#) => 1.0), 1, nothing)) => 0.05, (Point(5, true, OrderedDict{Point, Float64}(Point(#= circular reference @-2 =#) => 1.0), 1, nothing), Point(2, true, OrderedDict{Point, Float64}(Point(#= circular reference @-2 =#) => 1.0), 0, nothing)) => -0.05, (Point(2, true, OrderedDict{Point, Float64}(Point(#= circular reference @-2 =#) => 1.0), 0, nothing), Point(5, true, OrderedDict{Point, Float64}(Point(#= circular reference @-2 =#) => 1.0), 1, nothing)) => -0.05, (Point(2, true, OrderedDict{Point, Floa

### Describe the algorithm

Great, now it is time to implement the *fast gradient method* in a loop. 

In [20]:
κ = μ/L

x_new, y = x0, x0
for i in 1:n
    x_old = x_new
    x_new = y - 1 / L * gradient!(func, y)
    y = x_new + (1 - sqrt(κ)) / (1 + sqrt(κ)) * (x_new - x_old)
end

So when the loop ends `x_new` contains $x_n$. Let us now evaluate $f(x_n)$.

In [21]:
f_n = value!(func, x_new) # computes $f(x_n)$

Expression(32, true, OrderedDict{Any, Float64}(Expression(#= circular reference @-2 =#) => 1.0), 3, nothing)

Let us now tell the PEP that our performance measure is $f(x_N) - f_\star$.

In [22]:
set_performance_metric!(problem, value!(func, x_new) - fs) # set performance metric to $f(x_n) - f_\star$

1-element Vector{Expression}:
 Expression(35, false, OrderedDict{Any, Float64}(Expression(32, true, OrderedDict{Any, Float64}(Expression(#= circular reference @-2 =#) => 1.0), 3, nothing) => 1.0, Expression(4, true, OrderedDict{Any, Float64}(Expression(#= circular reference @-2 =#) => 1.0), 0, nothing) => -1.0), nothing, nothing)

### Solve the PEP
We have everything to build up the PEP. Now time to solve!

In [23]:
τ_PEPit = solve!(problem, solver = Mosek.Optimizer, verbose=true)

 💻 PEPit:  Setting up the problem: size of the main PSD matrix: 5x5
 💻 PEPit:  Setting up the problem: performance measure is minimum of 1 element(s)
 💻 PEPit:  Setting up the problem: Adding initial conditions and general constraints ...
 💻 PEPit:  Setting up the problem: initial conditions and general constraints (1 constraint(s) added)
 💻 PEPit:  Setting up the problem: interpolation conditions for 1 function(s)
		 function 1 : 12 scalar constraint(s) added
 💻 PEPit:  Compiling SDP
 💻 PEPit:  Calling SDP solver
Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 14              
  Affine conic cons.     : 0               
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 5               
  Matrix variables       : 1 (scalarized: 15)
  Integer variables      : 0               

Optimizer s

0.34760221803672986

### Comparison with theoretical results

If everything went well, we should get `τ_PEPit ≈ 0.34760221803672986` or something very close to it. What does the theory say in this regard? The following **upper** guarantee can be found in [1, Corollary 4.15]:

$$
f(x_n)-f_\star \leqslant \left(1 - \sqrt{\frac{\mu}{L}}\right)^n \left(f(x_0) -  f(x_\star) + \frac{\mu}{2}\|x_0 - x_\star\|^2\right).
$$. 

Let us compute the theoretical value of $\tau$.

In [24]:
τ_Theory = (1 - sqrt(κ))^n

@info "💻  PEPit guarantee: f(x_n)-f_*  <= $(round(τ_PEPit, digits=6)) (f(x_0) - f(x_*) + mu/2*||x_0 - x_*||^2)"

@info "📝 Theoretical guarantee: f(x_n)-f_*  <= $(round(τ_Theory, digits=6)) (f(x_0) - f(x_*) + mu/2*||x_0 - x_*||^2)"

[ Info: 💻  PEPit guarantee: f(x_n)-f_*  <= 0.347602 (f(x_0) - f(x_*) + mu/2*||x_0 - x_*||^2)
[ Info: 📝 Theoretical guarantee: f(x_n)-f_*  <= 0.467544 (f(x_0) - f(x_*) + mu/2*||x_0 - x_*||^2)


So the gurantee that we found via `PEPit` is tighter than the theoretical guarantee!

**References**:

[1] A. d’Aspremont, D. Scieur, A. Taylor (2021). Acceleration Methods. Foundations and Trends
in Optimization: Vol. 5, No. 1-2.